<a href="https://colab.research.google.com/github/Rakesh-2211/Monte-Carlo-BER-over-AWGN/blob/main/Experiment_12_%E2%80%94_Monte_Carlo_BER_over_AWGN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import erfc

def q_func(x):
    """Standard Q-function"""
    return 0.5 * erfc(x / np.sqrt(2))

class Modem:
    def __init__(self, scheme):
        self.scheme = scheme
        self.setup_constellation()

    def setup_constellation(self):
        if self.scheme == 'BPSK':
            self.k = 1
            self.constellation = np.array([-1, 1], dtype=complex)
            self.bit_map = {0: 0, 1: 1} # index to bit tuple (integer representation)
            self.theory_ber = lambda ebno: q_func(np.sqrt(2 * ebno))

        elif self.scheme == 'QPSK':
            self.k = 2
            self.constellation = np.array([-1-1j, -1+1j, 1-1j, 1+1j]) / np.sqrt(2)
            self.theory_ber = lambda ebno: q_func(np.sqrt(2 * ebno))

        elif self.scheme == 'BFSK':
            # Orthogonal coherent BFSK simulated in 2D real space
            self.k = 1
            self.constellation = np.array([[1, 0], [0, 1]]) # basis vectors
            self.theory_ber = lambda ebno: q_func(np.sqrt(ebno))

        elif self.scheme == '8-PSK':
            self.k = 3
            phases = np.array([0, 1, 3, 2, 6, 7, 5, 4]) * np.pi / 4 # Gray mapped phases
            self.constellation = np.exp(1j * phases)
            self.theory_ber = lambda ebno: (2/self.k) * q_func(np.sqrt(2 * self.k * ebno) * np.sin(np.pi/8))

        elif self.scheme == '16-QAM':
            self.k = 4
            pam = np.array([-3, -1, 3, 1]) # 1D Gray mapping
            X, Y = np.meshgrid(pam, pam)
            self.constellation = (X.flatten() + 1j * Y.flatten()) / np.sqrt(10)
            self.theory_ber = lambda ebno: (3/4) * q_func(np.sqrt((4/5) * ebno))

        elif self.scheme == '64-QAM':
            self.k = 6
            pam = np.array([-7, -5, -1, -3, 7, 5, 1, 3])
            X, Y = np.meshgrid(pam, pam)
            self.constellation = (X.flatten() + 1j * Y.flatten()) / np.sqrt(42)
            self.theory_ber = lambda ebno: (7/12) * q_func(np.sqrt((1/7) * ebno))
        else:
            raise ValueError("Unsupported modulation scheme")

        # Map indices to bit arrays for fast bit error counting
        self.M = 2**self.k
        self.int_to_bits = {i: np.array(list(np.binary_repr(i, width=self.k)), dtype=int) for i in range(self.M)}
        self.bit_matrix = np.array([self.int_to_bits[i] for i in range(self.M)])

    def modulate(self, bit_indices):
        return self.constellation[bit_indices]

    def demodulate(self, received_symbols):
        if self.scheme == 'BFSK':
            # 2D distance for orthogonal BFSK
            dists = np.linalg.norm(received_symbols[:, None, :] - self.constellation[None, :, :], axis=-1)
        else:
            # Complex distance for PSK/QAM
            dists = np.abs(received_symbols[:, None] - self.constellation[None, :])
        return np.argmin(dists, axis=1)

def run_monte_carlo_ber(schemes, ebno_db_range, max_errors=200, max_bits=1e6):
    results = {}

    for scheme in schemes:
        modem = Modem(scheme)
        ber_sim = []
        ber_theory = []
        err_counts = []
        bit_counts = []
        upper_bounds = []

        print(f"Simulating {scheme}...")

        for ebno_db in ebno_db_range:
            ebno_lin = 10**(ebno_db / 10)
            ber_theory.append(modem.theory_ber(ebno_lin))

            # Explicit Parameters
            bits_per_symbol = modem.k
            # Es/N0 = (Eb/N0) * k.  Normalized symbol energy E_s = 1.
            N0 = 1.0 / (ebno_lin * bits_per_symbol)
            noise_var_per_dim = N0 / 2

            total_errors = 0
            total_bits = 0
            chunk_size = max(1000, int(10000 / modem.k)) # Adaptive chunking

            # Adaptive driver stopping conditions
            while total_errors < max_errors and total_bits < max_bits:
                # Generate random symbol indices
                sym_indices = np.random.randint(0, modem.M, chunk_size)
                tx_bits = modem.bit_matrix[sym_indices]

                # Modulate
                tx_symbols = modem.modulate(sym_indices)

                # AWGN channel
                if scheme == 'BFSK':
                    noise = np.random.normal(0, np.sqrt(noise_var_per_dim), tx_symbols.shape)
                else:
                    noise = (np.random.normal(0, np.sqrt(noise_var_per_dim), len(tx_symbols)) +
                             1j * np.random.normal(0, np.sqrt(noise_var_per_dim), len(tx_symbols)))

                rx_symbols = tx_symbols + noise

                # Demodulate
                rx_indices = modem.demodulate(rx_symbols)
                rx_bits = modem.bit_matrix[rx_indices]

                # Count errors
                errors = np.sum(tx_bits != rx_bits)
                total_errors += errors
                total_bits += chunk_size * modem.k

            err_counts.append(total_errors)
            bit_counts.append(total_bits)

            if total_errors == 0:
                # Treat zero-error trials as upper bounds
                ber_sim.append(1.0 / total_bits)
                upper_bounds.append(True)
            else:
                ber_sim.append(total_errors / total_bits)
                upper_bounds.append(False)

        results[scheme] = {
            'ber_sim': np.array(ber_sim),
            'ber_theory': np.array(ber_theory),
            'err_counts': np.array(err_counts),
            'bit_counts': np.array(bit_counts),
            'upper_bounds': np.array(upper_bounds)
        }
    return results

# Configuration
schemes_to_run = ['BPSK', 'QPSK', 'BFSK', '8-PSK', '16-QAM', '64-QAM']
ebno_db_array = np.arange(0, 16, 2)

# Execute Simulation
sim_results = run_monte_carlo_ber(schemes_to_run, ebno_db_array)

# Visualizations
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

colors = ['b', 'g', 'r', 'c', 'm', 'k']
markers = ['o', 's', '^', 'D', 'v', 'p']

# Plot 1: Publication-quality semilog BER curves
for i, scheme in enumerate(schemes_to_run):
    res = sim_results[scheme]

    # Theory lines
    ax1.semilogy(ebno_db_array, res['ber_theory'], linestyle='-', color=colors[i], label=f'{scheme} (Theory)')

    # Simulation markers
    normal_idx = ~res['upper_bounds']
    ax1.semilogy(ebno_db_array[normal_idx], res['ber_sim'][normal_idx], marker=markers[i],
                 linestyle='', color=colors[i], label=f'{scheme} (Sim)')

    # Upper bounds (zero-error trials)[cite: 1]
    ub_idx = res['upper_bounds']
    if np.any(ub_idx):
        ax1.semilogy(ebno_db_array[ub_idx], res['ber_sim'][ub_idx], marker='v',
                     linestyle='', color=colors[i], markerfacecolor='none', label=f'{scheme} (Upper Bound)')

ax1.set_xlabel('Eb/N0 (dB)')
ax1.set_ylabel('Bit Error Rate (BER)')
ax1.set_title('Monte Carlo BER over AWGN')
ax1.set_ylim([1e-6, 1])
ax1.grid(True, which='both', linestyle='--')
ax1.legend(loc='lower left', fontsize='small', ncol=2)

# Plot 2: Error count versus transmitted bits[cite: 1]
for i, scheme in enumerate(schemes_to_run):
    res = sim_results[scheme]
    ax2.plot(ebno_db_array, res['err_counts'], marker=markers[i], linestyle='-', color=colors[i], label=f'{scheme} Errors')
    ax2.plot(ebno_db_array, res['bit_counts'] / 1000, marker=markers[i], linestyle='--', color=colors[i], alpha=0.5, label=f'{scheme} Bits (x1000)')

ax2.set_xlabel('Eb/N0 (dB)')
ax2.set_ylabel('Count')
ax2.set_title('Error Count & Transmitted Bits vs Eb/N0')
ax2.set_yscale('log')
ax2.grid(True, which='both', linestyle='--')
ax2.axhline(200, color='r', linestyle=':', label='Max Errors Target (200)')
ax2.axhline(1000000 / 1000, color='b', linestyle=':', label='Max Bits Target (10^6)')
ax2.legend(loc='lower left', fontsize='x-small', ncol=2)

plt.tight_layout()
plt.show()